# AERIS_64 · 1M training (Colab)
Runtime → **GPU**. SENTINEL engine · no-code corpus (Wiki / Gutenberg / literature).

In [ ]:
# Clone (private: insert PAT once, then revoke)
!git clone https://github.com/giridhari-code/AERIS_64.git
%cd AERIS_64
!pip -q install -e ".[train]" || pip -q install -e .


In [ ]:
# Build ~1M token pack
!PYTHONPATH=packages python scripts/data/build_1m_corpus.py
!ls -lh artifacts/dataset_1m/


In [ ]:
# Train SENTINEL on 1M corpus
!PYTHONPATH=packages python scripts/train/train_company.py \
  --data artifacts/dataset_1m/aeris64_1m_train.jsonl \
  --out checkpoints/aeris64_1m_ft \
  --steps 1200 \
  --preset small \
  --device cuda \
  --grad-accum 4 \
  --warmup 80 \
  --cosine \
  --lr 5e-4 \
  --product AERIS_64


In [ ]:
# Optional: fine-tune from example checkpoint instead of scratch
# !PYTHONPATH=packages python scripts/train/train_finetune.py \
#   --resume checkpoints/examples/AERIS_main \
#   --data artifacts/dataset_1m/aeris64_1m_train.jsonl \
#   --out checkpoints/aeris64_1m_ft \
#   --steps 800 --preset small --device cuda --grad-accum 4


In [ ]:
import os
os.environ["NEUROFIELD_CHECKPOINT"] = "checkpoints/aeris64_1m_ft"
os.environ["NEUROFIELD_DEVICE"] = "cuda"
print("Serve later:", "PYTHONPATH=packages python -m uvicorn sentinel.api.app:create_app --factory --host 0.0.0.0 --port 8000")
